# 03 · Fine-tuning and extending a deployed model (guided)

**Style A: syntax gaps.** 8 gaps (`# TODO(gN)` + `____`), each followed by a check.
Solutions: `../solutions/guided/03_finetune_extend.md`.

**Traces to** `notes/ioai-task-patterns.md` P4. T26-H-NIGHT had to grow a deployed classifier from 16 to 29
classes without forgetting (replay, distillation, freezing/LoRA, per-module LRs). T25-CHICKEN loaded a
**partial** state dict into a feature extractor. The syllabus lists "Model Finetuning (full and
parameter-efficient)".

Setup: a CNN "deployed" on FashionMNIST classes 0–4 must learn classes 5–9. Only **20 images per old
class** are left, plus 200 per new class. Score = ½·acc_old + ½·acc_new, the T26-H-NIGHT metric.
Runtime: about 1.5 minutes on a T4.

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torchvision import datasets
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

tr = datasets.FashionMNIST("data", train=True, download=True)
te = datasets.FashionMNIST("data", train=False, download=True)
X_all, y_all = tr.data.unsqueeze(1).float() / 255, tr.targets
X_test, y_test = te.data.unsqueeze(1).float() / 255, te.targets
OLD, NEW = list(range(5)), list(range(5, 10))

class Net(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU())
        self.head = nn.Linear(128, n_classes)
    def forward(self, x):
        return self.head(self.features(x))

def fit(model, X, y, epochs, lr, sampler=None, extra_loss=None, params=None):
    loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=sampler is None, sampler=sampler)
    opt = params if isinstance(params, torch.optim.Optimizer) else torch.optim.Adam(model.parameters() if params is None else params, lr=lr)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb) + (extra_loss(xb, logits) if extra_loss else 0)
            opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def score(model):
    model.eval()
    pred = torch.cat([model(X_test[i:i + 2000].to(device)).argmax(1).cpu() for i in range(0, 10000, 2000)])
    old, new = y_test < 5, y_test >= 5
    a_old, a_new = (pred[old] == y_test[old]).float().mean().item(), (pred[new] == y_test[new]).float().mean().item()
    return round(a_old, 3), round(a_new, 3), round(0.5 * a_old + 0.5 * a_new, 3)

# the "deployed" model: trained once on ALL old-class data, which we no longer have
old_mask = y_all < 5
deployed = Net(5).to(device)
fit(deployed, X_all[old_mask], y_all[old_mask], epochs=2, lr=1e-3)
torch.save(deployed.state_dict(), "deployed.pt")

g = torch.Generator().manual_seed(0)
def take(classes, k):
    idx = torch.cat([torch.nonzero(y_all == c).flatten()[torch.randperm(int((y_all == c).sum()), generator=g)[:k]] for c in classes])
    return X_all[idx], y_all[idx]
X_old, y_old = take(OLD, 20)          # the small retained subset of old data
X_new, y_new = take(NEW, 200)         # the new customer's data
print("deployed 5-class model (old acc, new acc, score):", score(deployed))

## 1. Load the deployed weights into a 10-class model

The backbone matches, but the head has a different shape (5 → 10 rows). Load everything except the head, then
copy the 5 old rows into the new head so the old classes keep working on day one.

In [ ]:
state = torch.load("deployed.pt", map_location=device)
model = Net(10).to(device)
backbone_state = {k: v for k, v in state.items() if not k.startswith("head.")}
# TODO(g1): load a state dict that does NOT contain every key of the model (it returns missing/unexpected keys)
res = ____
with torch.no_grad():
    # TODO(g2): copy the old head's weight and bias into rows 0..4 of the new head
    ____

In [ ]:
assert sorted(res.missing_keys) == ["head.bias", "head.weight"] and not res.unexpected_keys
x = X_test[:8].to(device); deployed.eval(); model.eval()
assert torch.allclose(model(x)[:, :5], deployed(x), atol=1e-5), "old logits changed"
print("g1, g2 ok | day-one score (old, new, total):", score(model))

## 2. The naive approach: fine-tune on the new data only

Watch the old accuracy collapse (catastrophic forgetting). T26-H-NIGHT asked you to "watch it happen at
least once".

In [ ]:
naive = copy.deepcopy(model)
fit(naive, X_new, y_new, epochs=3, lr=1e-3)
print("naive fine-tune (old, new, total):", score(naive))

## 3. Freeze the backbone, train only the head

In [ ]:
frozen = copy.deepcopy(model)
# TODO(g3): stop gradients for every parameter of frozen.features
for p in frozen.features.parameters():
    ____
n_train = sum(p.numel() for p in frozen.parameters() if p.requires_grad)
fit(frozen, torch.cat([X_old, X_new]), torch.cat([y_old, y_new]), epochs=5, lr=1e-3,
    params=[p for p in frozen.parameters() if p.requires_grad])
print(f"head only ({n_train} trainable params):", score(frozen))

In [ ]:
assert n_train == 128 * 10 + 10
assert all(not p.requires_grad for p in frozen.features.parameters())
print("g3 ok")

## 4. Replay + class-balanced sampling + distillation + two learning rates

- **Replay**: mix the 100 retained old images in with the new ones.
- **Balanced sampling**: 100 old vs 1000 new images, so sample each image with weight 1/(its class count).
- **Knowledge distillation**: keep the old-class logits close to the frozen deployed model's (Hinton KD:
  KL between temperature-softened distributions, scaled by T²).
- **Per-module learning rates**: a small LR for the backbone, a larger one for the head.

In [ ]:
X_mix, y_mix = torch.cat([X_old, X_new]), torch.cat([y_old, y_new])
counts = torch.bincount(y_mix, minlength=10).float()
# TODO(g4): one weight per SAMPLE = 1 / (number of samples of its class)
weights = ____
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

teacher = copy.deepcopy(deployed)
# TODO(g5): the teacher is a fixed target: inference mode and no gradients
____

T = 2.0
def kd_term(xb, logits):
    with torch.no_grad():
        t_logits = teacher(xb)
    # TODO(g6): KL(teacher || student) on the OLD-class logits, temperature T, scaled by T*T
    return ____

In [ ]:
lg = torch.randn(4, 10); tl = lg[:, :5].clone()
teacher_backup = teacher; teacher = lambda x: tl                      # a fake teacher that agrees exactly
assert abs(kd_term(None, lg).item()) < 1e-5, "KD must be ~0 when student and teacher agree"
teacher = teacher_backup
assert np.isclose(weights[y_mix == 0].sum().item(), 1.0) and np.isclose(weights[y_mix == 7].sum().item(), 1.0)
assert not teacher.training and not any(p.requires_grad for p in teacher.parameters())
print("g4, g5, g6 ok")

In [ ]:
student = copy.deepcopy(model)
# TODO(g7): two parameter groups: features at lr 1e-4, head at lr 1e-3
opt = torch.optim.Adam(____)
fit(student, X_mix, y_mix, epochs=6, lr=None, sampler=sampler, extra_loss=kd_term, params=opt)
print("replay + balanced + KD + two LRs:", score(student))

In [ ]:
assert [g["lr"] for g in opt.param_groups] == [1e-4, 1e-3]
assert score(student)[2] > score(naive)[2]
print("g7 ok")

## 5. Parameter-efficient fine-tuning with `peft` (LoRA)

LoRA freezes a `Linear` layer and learns a low-rank update ΔW = B·A (rank r). `peft` wraps any `nn.Module`:
name the target layers, and list the modules you still want fully trainable in `modules_to_save`.

In [ ]:
from peft import LoraConfig, get_peft_model
base = copy.deepcopy(model)
print([name for name, m in base.named_modules() if isinstance(m, nn.Linear)])
# TODO(g8): LoRA rank 8 (alpha 16) on the backbone's Linear "features.9"; keep "head" fully trainable
config = LoraConfig(____)
peft_model = get_peft_model(base, config)
peft_model.print_trainable_parameters()
fit(peft_model, X_mix, y_mix, epochs=6, lr=1e-3, sampler=sampler, extra_loss=kd_term,
    params=[p for p in peft_model.parameters() if p.requires_grad])
print("LoRA + head:", score(peft_model))

In [ ]:
n_lora = sum(p.numel() for n, p in peft_model.named_parameters() if p.requires_grad and "lora_" in n)
assert n_lora == 8 * (3136 + 128), n_lora
print("g8 ok, all gaps done")